# Probe transfer: activation extraction and analysis
Runtime > Change runtime type > **A100** (Colab Pro). On a T4 the 32B pair will not fit.

Before running: put `extract.py`, `analyze.py` and `run_colab.sh` in the folder below, next to this notebook. Everything is read from and written to that folder; nothing needs re-uploading after a disconnect.

In [11]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [7]:
!pip -q install transformers accelerate bitsandbytes scikit-learn scipy pandas

In [12]:
import os
from google.colab import userdata
tok = userdata.get("HF_TOKEN_READ")
os.environ["HF_TOKEN"] = tok
os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import login, whoami
login(token=tok, add_to_git_credential=False)
print("logged in as", whoami()["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in as MariosTsatsos


In [16]:
D = "/content/drive/MyDrive/Work/Mech Interp/BlueDot Technical Project - P4/notebooks"
!curl -sSL -o "$D/xstest_prompts.csv" https://raw.githubusercontent.com/paul-rottger/xstest/main/xstest_prompts.csv
import pandas as pd; d = pd.read_csv(f"{D}/xstest_prompts.csv"); print(d.shape, d.label.value_counts().to_dict())

(450, 6) {'safe': 250, 'unsafe': 200}


In [19]:
!rm -rf ~/.cache/huggingface

In [20]:
D = "/content/drive/MyDrive/Work/Mech Interp/BlueDot Technical Project - P4/notebooks"
!bash "$D/run_colab.sh" "$D/xstest_prompts.csv" "$D"

Streaming output truncated to the last 5000 lines.
Reconstructing (incomplete total...):  79% 24.5G/31.2G [02:22<00:43, 155MB/s]
Reconstructing (incomplete total...):  89% 27.9G/31.2G [02:39<00:19, 167MB/s]
Reconstructing (incomplete total...):  95% 29.6G/31.2G [02:49<00:10, 158MB/s]

Fetching 17 files:   6% 1/17 [02:52<45:56, 172.31s/it]

Fetching 17 files:  12% 2/17 [02:52<17:45, 71.05s/it] 
Reconstructing (incomplete total...):  86% 30.2G/35.1G [02:52<00:24, 202MB/s]
Reconstructing (incomplete total...):  77% 30.2G/39.0G [02:52<00:43, 200MB/s]

Fetching 17 files:  18% 3/17 [02:52<09:01, 38.71s/it]
Reconstructing (incomplete total...):  71% 30.3G/42.9G [02:52<01:02, 202MB/s]

Fetching 17 files:  24% 4/17 [02:53<05:08, 23.74s/it]
Reconstructing (incomplete total...):  65% 30.4G/46.8G [02:53<01:26, 190MB/s]
Reconstructing (incomplete total...):  60% 30.4G/50.7G [02:53<01:46, 190MB/s]

Fetching 17 files:  35% 6/17 [02:56<02:12, 12.02s/it]
Reconstructing (incomplete total...):  57% 31.0G

Results land in the same folder: `acts_*.npz` (activations, keep these), `*_summary.json`, `*_folds.csv`, `*_log.txt`. If the session disconnects, rerun the last cell; finished models are skipped.

In [13]:
D = "/content/drive/MyDrive/Work/Mech Interp/BlueDot Technical Project - P4/notebooks"
!bash "$D/run_extra.sh" "$D/xstest_prompts.csv" "$D"

math-7b: layer 18
   direction  n_pos  n_neg  assoc_with_safety  cos_full  cos_cross  ci_lo  ci_hi  floor  rotation
      safety    200    250              1.000    0.8260     0.6159 0.5267 0.6784 0.7288    0.1129
question_how    178    272              0.228    0.8370     0.7509 0.6969 0.7816 0.8903    0.1394
named_entity    138    312              0.197    0.8276     0.7027 0.6186 0.7493 0.8386    0.1359
 length_long    217    233              0.085    0.6548     0.6163 0.5893 0.6361 0.9410    0.3247
type splits (n=35): rotation median 0.173, range 0.121 to 0.231; 35 of 35 rotate MORE than the safety direction (0.113)
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 62.8MB/s]
model-00004-of-00004.safetensors: 100% 3.56G/3.56G [00:27<00:00, 128MB/s]
config.json: 100% 678/678 [00:00<00:00, 4.09MB/s]
tokenizer_config.json: 100% 7.32k/7.32k [00:00<00:00, 14.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 11.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 2.13MB/s]
tokeniz